## Install dependencies

In [1]:
!pip install -q -U chromadb sentence-transformers
!pip install -q -U transformers accelerate bitsandbytes

## Load the sample dataset


In [2]:
import pandas as pd
df = pd.read_csv("/content/telecom_logs.csv")
df.head()

,Region,Tower_ID,Date,Call_Drops,Signal_Strength_dBm,Congestion_Level,Handoff_Failure_Percent,Notes
0,Bangalore,T321,2025-09-01,46,-87,High,10.1,Office-hours congestion
1,Bangalore,T321,2025-09-02,54,-90,High,10.4,Weekend event nearby
2,Bangalore,T321,2025-09-03,41,-86,Medium,6.9,Stable performance
3,Bangalore,T321,2025-09-04,40,-85,High,9.3,Office-hours congestion
4,Bangalore,T321,2025-09-05,49,-91,High,7.1,Recent equipment upgrade


## Clean, chunk, embed, and store in ChromaDB

In [3]:
# ---- Clean ----
df = df.dropna(subset=["Region", "Tower_ID", "Call_Drops"])
for col in ["Call_Drops", "Signal_Strength_dBm", "Handoff_Failure_Percent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Notes"] = df["Notes"].fillna("No additional notes")

In [4]:
# ---- Chunk: turn each row into a readable sentence ----
def row_to_text(row):
    return (
        f"On {row['Date']}, tower {row['Tower_ID']} in {row['Region']} recorded "
        f"{row['Call_Drops']} call drops. Signal strength was {row['Signal_Strength_dBm']} dBm, "
        f"congestion level was {row['Congestion_Level']}, and handoff failure rate was "
        f"{row['Handoff_Failure_Percent']}%. Notes: {row['Notes']}."
    )
df["chunk_text"] = df.apply(row_to_text, axis=1)

df.head()

,Region,Tower_ID,Date,Call_Drops,Signal_Strength_dBm,Congestion_Level,Handoff_Failure_Percent,Notes,chunk_text
0,Bangalore,T321,2025-09-01,46,-87,High,10.1,Office-hours congestion,"On 2025-09-01, tower T321 in Bangalore recorde..."
1,Bangalore,T321,2025-09-02,54,-90,High,10.4,Weekend event nearby,"On 2025-09-02, tower T321 in Bangalore recorde..."
2,Bangalore,T321,2025-09-03,41,-86,Medium,6.9,Stable performance,"On 2025-09-03, tower T321 in Bangalore recorde..."
3,Bangalore,T321,2025-09-04,40,-85,High,9.3,Office-hours congestion,"On 2025-09-04, tower T321 in Bangalore recorde..."
4,Bangalore,T321,2025-09-05,49,-91,High,7.1,Recent equipment upgrade,"On 2025-09-05, tower T321 in Bangalore recorde..."


In [5]:
# ---- Embed ----
from sentence_transformers import SentenceTransformer
print("Loading embedding model (all-MiniLM-L6-v2)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(df["chunk_text"].tolist()).tolist()

Loading embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# ---- Store in ChromaDB ----
import chromadb
client = chromadb.PersistentClient(path="/content/chroma_db")
try:
    client.delete_collection("telecom_logs")
except Exception:
    pass
collection = client.create_collection("telecom_logs")

ids = [f"log_{i}" for i in range(len(df))]
metadatas = df[[
    "Region", "Tower_ID", "Date", "Call_Drops",
    "Signal_Strength_dBm", "Congestion_Level", "Handoff_Failure_Percent"
]].to_dict(orient="records")

collection.add(ids=ids, embeddings=embeddings, documents=df["chunk_text"].tolist(), metadatas=metadatas)
print(f"Stored {len(df)} log entries in ChromaDB.")


Stored 224 log entries in ChromaDB.


## The underlying tool logic (RAG retrieval + rule-based reasoning)



In [7]:
def get_all_logs_for_region(region: str, tower_id: str = None):
    """Tool 1: fetch the FULL set of log rows for a region (optionally
    narrowed to one tower) via an exact metadata filter.
     Also sorted chronologically so trend comparisons are meaningful."""
    where = {"Region": region} if tower_id is None else {
        "$and": [{"Region": region}, {"Tower_ID": tower_id}]
    }
    results = collection.get(where=where)
    if not results["documents"]:
        return []
    logs = [{"text": doc, "meta": meta} for doc, meta in zip(results["documents"], results["metadatas"])]
    logs.sort(key=lambda l: l["meta"]["Date"])  # chronological, so drops[0]→drops[-1] is a real trend
    return logs

In [8]:
def summarize_issue(region: str, logs: list) -> dict:
    """Tool 2 (fact-finding half): work out observation + root cause with explainable rules."""
    if not logs:
        return {"observation": f"No log data found for {region}.",
                "root_cause": "Unable to determine - no data available.", "signals": {}}
    drops = [l["meta"]["Call_Drops"] for l in logs]
    signals_dbm = [l["meta"]["Signal_Strength_dBm"] for l in logs]
    handoff_fail = [l["meta"]["Handoff_Failure_Percent"] for l in logs]
    congestion_levels = [l["meta"]["Congestion_Level"] for l in logs]
    avg_signal = sum(signals_dbm) / len(signals_dbm)
    avg_handoff = sum(handoff_fail) / len(handoff_fail)
    most_common_congestion = max(set(congestion_levels), key=congestion_levels.count)

    if len(drops) > 1 and drops[-1] > drops[0]:
        observation = f"Call drops increased from {drops[0]} to {drops[-1]} across recent logs."
    elif len(drops) > 1 and drops[-1] < drops[0]:
        observation = f"Call drops decreased from {drops[0]} to {drops[-1]} across recent logs."
    else:
        observation = f"Call drops have stayed roughly steady at around {drops[0]}."

    causes = []
    if avg_signal <= -90:
        causes.append(f"weak average signal strength ({avg_signal:.0f} dBm)")
    if most_common_congestion == "High":
        causes.append("high tower congestion")
    if avg_handoff >= 10:
        causes.append(f"elevated handoff failure rate ({avg_handoff:.1f}%)")
    root_cause = " + ".join(causes) if causes else "No major issues detected - metrics are within normal range."

    return {"observation": observation, "root_cause": root_cause,
            "signals": {"avg_signal_dbm": round(avg_signal, 1),
                        "avg_handoff_failure_pct": round(avg_handoff, 1),
                        "congestion_level": most_common_congestion}}

In [9]:
def recommend_resolution(signals: dict) -> list:
    """Tool 3: rule-based resolution suggestions."""
    if not signals:
        return ["No recommendation - insufficient data."]
    recs = []
    if signals.get("congestion_level") == "High":
        recs.append("Deploy an additional microcell/small cell during peak hours.")
        recs.append("Increase backhaul capacity at the affected tower.")
    if signals.get("avg_handoff_failure_pct", 0) >= 10:
        recs.append("Optimize handoff algorithms and adjust handover thresholds.")
    if signals.get("avg_signal_dbm", 0) <= -90:
        recs.append("Perform a signal audit; consider antenna tilt/height adjustment or a signal booster.")
    if not recs:
        recs.append("Continue routine monitoring - no urgent action needed.")
    return recs


## Load Mistral-7B-Instruct-v0.2 in 4-bit


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline as hf_pipeline_fn

In [11]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# Guard against accidentally loading the model twice if you re-run this cell
if "llm_pipeline" not in globals():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"Downloading and loading {MODEL_ID} in 4-bit (this can take several minutes)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )

    llm_pipeline = hf_pipeline_fn(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False,
    clean_up_tokenization_spaces=False,
    )
    llm_pipeline.model.generation_config.max_length = None
    llm_pipeline.model.generation_config.temperature = None
    llm_pipeline.model.generation_config.do_sample = False
    print("Model loaded.")
else:
    print("Model already loaded, skipping reload.")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded.


## Define the Tools


In [12]:
def clean_region_input(text: str) -> str:
    """The LLM sometimes wraps its tool input in quotes or extra words - clean it up."""
    text = text.strip().strip('"').strip("'")
    for region in ["Hyderabad", "Delhi", "Mumbai", "Bangalore", "Chennai"]:
        if region.lower() in text.lower():
            return region
    return text

In [13]:
def tool_query_logs(region: str) -> str:
    region = clean_region_input(region)
    logs = get_all_logs_for_region(region)
    if not logs:
        return f"No logs found for region '{region}'."
    return "\n".join(l["text"] for l in logs)

In [14]:
def tool_summarize(region: str) -> str:
    region = clean_region_input(region)
    logs = get_all_logs_for_region(region)
    summary = summarize_issue(region, logs)
    return f"Observation: {summary['observation']} Root cause: {summary['root_cause']}"

In [15]:
def tool_recommend(region: str) -> str:
    region = clean_region_input(region)
    logs = get_all_logs_for_region(region)
    summary = summarize_issue(region, logs)
    recs = recommend_resolution(summary["signals"])
    return "; ".join(recs)

In [16]:
# Plain-Python tool registry, since the LLM still reads these to decide which tool to call.
TOOLS = {
    "QueryLogs": {
        "func": tool_query_logs,
        "description": "Fetches relevant telecom log snippets for a given region. "
                        "Input should be ONLY the region name, e.g. 'Hyderabad'.",
    },
    "SummarizeIssue": {
        "func": tool_summarize,
        "description": "Returns the observed call-drop trend and the likely root cause "
                        "for a given region, based on signal strength, congestion, and "
                        "handoff data. Input should be ONLY the region name.",
    },
    "RecommendResolution": {
        "func": tool_recommend,
        "description": "Returns recommended fixes for the call-drop issue in a given "
                        "region. Input should be ONLY the region name.",
    },
}

## Build the ReAct Agent


This is the actual agent: given a question, the LLM reasons step
by step about which tool to call, reads the tool's output, and decides what to do next until it can give a final answer.


In [17]:
import re

In [18]:
MAX_STEPS = 5  # cap on Thought/Action/Observation cycles

REACT_PROMPT = """Answer the following question as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question, including root cause, evidence, and resolution steps

Begin!

Question: {question}
Thought:{scratchpad}"""

ACTION_RE = re.compile(r"Action:\s*(.+)")
ACTION_INPUT_RE = re.compile(r"Action Input:\s*(.+)")
FINAL_ANSWER_RE = re.compile(r"Final Answer:\s*(.+)", re.DOTALL)

In [19]:
def _tools_block() -> str:
    return "\n".join(f"{name}: {t['description']}" for name, t in TOOLS.items())

In [20]:
def _generate(prompt: str) -> str:
    """One forward pass through the raw HF pipeline."""
    output = llm_pipeline(prompt)[0]["generated_text"]
    # The model sometimes keeps talking past its turn and hallucinates its
    # own Observation — cut there so we can supply the *real* one.
    return output.split("\nObservation:")[0].strip()

In [21]:
def run_react_agent(question: str) -> str:
    """Manual ReAct loop: Thought -> Action -> Action Input -> Observation,
    repeated until the model emits 'Final Answer:' or MAX_STEPS is hit."""
    scratchpad = ""
    for _ in range(MAX_STEPS):
        prompt = REACT_PROMPT.format(
            tools=_tools_block(),
            tool_names=", ".join(TOOLS.keys()),
            question=question,
            scratchpad=scratchpad,
        )
        step_text = _generate(prompt)

        final_match = FINAL_ANSWER_RE.search(step_text)
        if final_match:
            return final_match.group(1).strip()

        action_match = ACTION_RE.search(step_text)
        input_match = ACTION_INPUT_RE.search(step_text)
        if not (action_match and input_match):
            raise ValueError(f"Could not parse agent step:\n{step_text}")

        action = action_match.group(1).strip()
        action_input = input_match.group(1).strip()
        if action not in TOOLS:
            raise ValueError(f"Model requested unknown tool '{action}'")

        observation = TOOLS[action]["func"](action_input)
        scratchpad += f"{step_text}\nObservation: {observation}\nThought:"

    raise ValueError("Max steps reached without a Final Answer.")

In [22]:
KNOWN_REGIONS = ["Hyderabad", "Delhi", "Mumbai", "Bangalore", "Chennai"]

def extract_region(user_query: str):
    for region in KNOWN_REGIONS:
        if re.search(region, user_query, re.IGNORECASE):
            return region
    return None

In [23]:
def deterministic_fallback_report(user_query: str) -> str:
    region = extract_region(user_query)
    if not region:
        return f"Couldn't find a known region in the question. Try one of: {', '.join(KNOWN_REGIONS)}."
    logs = get_all_logs_for_region(region)
    summary = summarize_issue(region, logs)
    recs = recommend_resolution(summary["signals"])
    lines = [f"Region: {region}", f"Observation: {summary['observation']}",
             f"Root Cause: {summary['root_cause']}", "Suggested Resolution:"]
    lines += [f"   {i}. {r}" for i, r in enumerate(recs, 1)]
    lines.append("\nEvidence from logs:")
    lines += [f"  - {l['text']}" for l in logs]
    return "\n".join(lines)

In [24]:
def run_strict_agent(query: str) -> str:
    """Runs the ReAct agent; falls back to the deterministic pipeline if it fails."""
    try:
        return run_react_agent(query)
    except Exception as e:
        print(f"[Agent parsing failed, using fallback: {e}]\n")
        return deterministic_fallback_report(query)

## Test use cases

In [25]:
from IPython.display import display, Markdown

In [26]:
answer = run_strict_agent("Why are call drops happening in Hyderabad?")
print("\nFINAL REPORT:")
display(Markdown(answer))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length


FINAL REPORT:


The call drops in Hyderabad are primarily due to weak signal strength, high tower congestion, and elevated handoff failure rate. To resolve this issue, consider deploying an additional microcell/small cell during peak hours, increasing backhaul capacity at the affected tower, optimizing handoff algorithms and adjusting handover thresholds, performing a signal audit, and considering antenna tilt/height adjustment or a signal booster for tower T124.

In [27]:
answer = run_strict_agent("Suggest fixes for call drops in Hyderabad.")
print("\nFINAL REPORT:")
display(Markdown(answer))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL REPORT:


The call drops in Hyderabad are due to weak signal strength, high tower congestion, and elevated handoff failure rate. To address this issue, consider deploying an additional microcell/small cell during peak hours, increasing backhaul capacity at the affected tower, optimizing handoff algorithms and adjusting handover thresholds, performing a signal audit, and considering antenna tilt/height adjustment or a signal booster.

In [28]:
answer = run_strict_agent("Is it congestion the cause for call drop for pune?")
print("\nFINAL REPORT:")
display(Markdown(answer))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL REPORT:


Based on the analysis, it appears that congestion is not the cause for call drops in Pune. The call drop trend has decreased, and no major issues were detected in the recent logs.

In [29]:
answer = run_strict_agent("What is causing call drops in Chennai?")
print("\nFINAL REPORT:")
display(Markdown(answer))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been 

[Agent parsing failed, using fallback: Max steps reached without a Final Answer.]


FINAL REPORT:


Region: Chennai
Observation: Call drops increased from 14 to 27 across recent logs.
Root Cause: No major issues detected - metrics are within normal range.
Suggested Resolution:
   1. Continue routine monitoring - no urgent action needed.

Evidence from logs:
  - On 2025-09-01, tower T654 in Chennai recorded 14 call drops. Signal strength was -66 dBm, congestion level was Low, and handoff failure rate was 3.4%. Notes: Standard load conditions.
  - On 2025-09-01, tower T655 in Chennai recorded 25 call drops. Signal strength was -75 dBm, congestion level was Medium, and handoff failure rate was 3.7%. Notes: No anomalies reported.
  - On 2025-09-02, tower T654 in Chennai recorded 24 call drops. Signal strength was -77 dBm, congestion level was High, and handoff failure rate was 0.9%. Notes: Office-hours congestion.
  - On 2025-09-02, tower T655 in Chennai recorded 17 call drops. Signal strength was -75 dBm, congestion level was Medium, and handoff failure rate was 2.5%. Notes: Normal operation.
  - On 2025-09-03, tower T654 in Chennai recorded 15 call drops. Signal strength was -75 dBm, congestion level was Low, and handoff failure rate was 3.1%. Notes: Standard load conditions.
  - On 2025-09-03, tower T655 in Chennai recorded 15 call drops. Signal strength was -73 dBm, congestion level was Medium, and handoff failure rate was 0.6%. Notes: Standard load conditions.
  - On 2025-09-04, tower T654 in Chennai recorded 4 call drops. Signal strength was -67 dBm, congestion level was Low, and handoff failure rate was 0.5%. Notes: Normal operation.
  - On 2025-09-04, tower T655 in Chennai recorded 27 call drops. Signal strength was -70 dBm, congestion level was High, and handoff failure rate was 2.6%. Notes: No anomalies reported.
  - On 2025-09-05, tower T654 in Chennai recorded 9 call drops. Signal strength was -77 dBm, congestion level was Medium, and handoff failure rate was 0.5%. Notes: Backhaul link degraded.
  - On 2025-09-05, tower T655 in Chennai recorded 13 call drops. Signal strength was -76 dBm, congestion level was Low, and handoff failure rate was 3.2%. Notes: Routine traffic.
  - On 2025-09-06, tower T654 in Chennai recorded 3 call drops. Signal strength was -68 dBm, congestion level was Low, and handoff failure rate was 1.7%. Notes: Standard load conditions.
  - On 2025-09-06, tower T655 in Chennai recorded 7 call drops. Signal strength was -75 dBm, congestion level was Medium, and handoff failure rate was 0.8%. Notes: Stable performance.
  - On 2025-09-07, tower T654 in Chennai recorded 21 call drops. Signal strength was -72 dBm, congestion level was High, and handoff failure rate was 0.5%. Notes: Standard load conditions.
  - On 2025-09-07, tower T655 in Chennai recorded 14 call drops. Signal strength was -75 dBm, congestion level was Medium, and handoff failure rate was 1.7%. Notes: Normal operation.
  - On 2025-09-08, tower T654 in Chennai recorded 13 call drops. Signal strength was -69 dBm, congestion level was Low, and handoff failure rate was 0.5%. Notes: Normal operation.
  - On 2025-09-08, tower T655 in Chennai recorded 23 call drops. Signal strength was -78 dBm, congestion level was High, and handoff failure rate was 3.2%. Notes: Weekend event nearby.
  - On 2025-09-09, tower T654 in Chennai recorded 9 call drops. Signal strength was -66 dBm, congestion level was Medium, and handoff failure rate was 0.5%. Notes: Routine traffic.
  - On 2025-09-09, tower T655 in Chennai recorded 20 call drops. Signal strength was -74 dBm, congestion level was Medium, and handoff failure rate was 1.3%. Notes: No anomalies reported.
  - On 2025-09-10, tower T654 in Chennai recorded 5 call drops. Signal strength was -70 dBm, congestion level was Low, and handoff failure rate was 1.8%. Notes: No anomalies reported.
  - On 2025-09-10, tower T655 in Chennai recorded 15 call drops. Signal strength was -72 dBm, congestion level was Medium, and handoff failure rate was 2.3%. Notes: Standard load conditions.
  - On 2025-09-11, tower T654 in Chennai recorded 9 call drops. Signal strength was -70 dBm, congestion level was Medium, and handoff failure rate was 0.5%. Notes: Standard load conditions.
  - On 2025-09-11, tower T655 in Chennai recorded 9 call drops. Signal strength was -77 dBm, congestion level was Low, and handoff failure rate was 4.0%. Notes: Normal operation.
  - On 2025-09-12, tower T654 in Chennai recorded 11 call drops. Signal strength was -75 dBm, congestion level was Medium, and handoff failure rate was 1.4%. Notes: Stable performance.
  - On 2025-09-12, tower T655 in Chennai recorded 6 call drops. Signal strength was -71 dBm, congestion level was Low, and handoff failure rate was 0.9%. Notes: Standard load conditions.
  - On 2025-09-13, tower T654 in Chennai recorded 12 call drops. Signal strength was -76 dBm, congestion level was Medium, and handoff failure rate was 1.3%. Notes: Stable performance.
  - On 2025-09-13, tower T655 in Chennai recorded 16 call drops. Signal strength was -70 dBm, congestion level was High, and handoff failure rate was 0.5%. Notes: Heavy user load.
  - On 2025-09-14, tower T654 in Chennai recorded 3 call drops. Signal strength was -67 dBm, congestion level was Low, and handoff failure rate was 0.5%. Notes: Routine traffic.
  - On 2025-09-14, tower T655 in Chennai recorded 27 call drops. Signal strength was -73 dBm, congestion level was High, and handoff failure rate was 2.5%. Notes: New residential area load.

In [30]:
answer = run_strict_agent("Suggest fixes for tower congestion in Delhi.")
print("\nFINAL REPORT:\n")
display(Markdown(answer))


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL REPORT:



The root cause of call drops in Delhi is high tower congestion. To mitigate this issue, consider deploying an additional microcell/small cell during peak hours and increasing backhaul capacity at the affected tower.